In [3]:
# ══════════════════════════════════════════════════════════════
#  셀 0/5  —  Drive 마운트 + 패키지 설치 + 경로 설정  [Rev5]
# ══════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import subprocess
subprocess.run(['pip', 'install', '-q', 'peft', 'accelerate'], check=True)

DRIVE_BASE   = '/content/drive/MyDrive/X-MultiVLA_rev5'
DATA_DIR     = f'{DRIVE_BASE}/data'
CKPT_DIR     = f'{DRIVE_BASE}/checkpoints'
DATASET_FILE = f'{DATA_DIR}/dataset_8hr_full.parquet'
HF_CACHE     = f'{DRIVE_BASE}/hf_cache'

import os, sys, torch
sys.path.insert(0, DRIVE_BASE)
import importlib; importlib.invalidate_caches()
sys.path_importer_cache.clear()

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE
os.environ['HF_HOME'] = HF_CACHE

# ── GPU 확인 + A100/H100 최적화 ──────────────────────────────
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU : {gpu_name}  ({vram_gb:.0f} GB)')
    # A100 / H100 은 TF32 가속 활성화
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    print('  TF32 enabled (matmul + cudnn)')
    USE_BF16 = torch.cuda.is_bf16_supported()
    print(f'  BF16 support : {USE_BF16}')
else:
    gpu_name = 'CPU'
    USE_BF16 = False
    print('GPU 없음 — CPU 모드')

print(f'Drive 마운트 완료')
print(f'  Dataset     : {DATASET_FILE}')
print(f'  Checkpoints : {CKPT_DIR}')
print(f'  HF Cache    : {HF_CACHE}')


Mounted at /content/drive
GPU : NVIDIA A100-SXM4-40GB  (42 GB)
  TF32 enabled (matmul + cudnn)
  BF16 support : True
Drive 마운트 완료
  Dataset     : /content/drive/MyDrive/X-MultiVLA_rev5/data/dataset_8hr_full.parquet
  Checkpoints : /content/drive/MyDrive/X-MultiVLA_rev5/checkpoints
  HF Cache    : /content/drive/MyDrive/X-MultiVLA_rev5/hf_cache


In [4]:
# ══════════════════════════════════════════════════════════════
#  셀 1/5  —  Config + 모델 정의  [v16]
#  ① iTransformer (자산 축 반전 어텐션)
#  ② NewsEncoder: 감성 스칼라 → l_emb
#  ③ BiCrossAttn: V→L + L→V 양방향 융합
#  ④ LLMReasoningModule: Qwen2.5-1.5B + LoRA (Phase2 전용)
#  ⑤ Phase1Head (빠른 SFT) / ActionHead (Phase2)
#  ⑥ PriceHead: MIN/MAX 수익률 예측
# ══════════════════════════════════════════════════════════════
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd

# ── Config ────────────────────────────────────────────────────
SYM_MAP  = {'BTC-USD':'BTC','ETH-USD':'ETH','SOL-USD':'SOL',
             'XRP-USD':'XRP','DOGE-USD':'DOGE'}
ASSETS   = ['BTC','ETH','SOL','XRP','DOGE']
N_ASSETS = 5
SEQ_LEN   = 60
D_MODEL   = 256
N_HEADS   = 8
N_LAYERS  = 2
DROPOUT   = 0.1
BATCH_SIZE      = 64   # A100 40GB 기준 (FP32 기준 32, BF16 AMP 기준 64)
GRPO_BATCH_SIZE = 16   # Qwen 포함, BF16 AMP 기준
COMMISSION = 0.001
TRAIN_START     = '2024-06-01'
LLM_NAME        = 'Qwen/Qwen2.5-1.5B'
LORA_RANK       = 16
LORA_ALPHA      = 32
NEWS_N_FEAT          = 4
NEWS_FEATURE_NAMES   = ['pos', 'neg', 'neu', 'score']
LLM_MAX_NEWS    = 3
SYSTEM_PROMPT   = ("You are a crypto portfolio manager. "
                   "Analyze market state and news to decide allocation.")

ROUNDS = [
    {'name':'R1','test_start':'2025-03-01','test_end':'2025-06-01'},
    {'name':'R2','test_start':'2025-06-01','test_end':'2025-09-01'},
    {'name':'R3','test_start':'2025-09-01','test_end':'2025-12-01'},
    {'name':'R4','test_start':'2025-12-01','test_end':'2026-03-01'},
    {'name':'R5','test_start':'2026-03-01','test_end':'2026-06-01'},
]

device    = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE    = device
WF_ROUNDS = 10
ROUND_WEEKS = 1
LR        = 3e-4
EPOCHS_P1 = 80
GRPO_LR   = 1e-5
EPOCHS_G  = 20
HORIZON   = 12
N_COINS   = N_ASSETS

# AMP dtype: A100/H100 → BF16, 그 외 GPU → FP16, CPU → FP32
AMP_DTYPE = (torch.bfloat16 if USE_BF16
             else torch.float16 if device == 'cuda'
             else torch.float32)
USE_AMP   = device == 'cuda'
print(f'Device: {device}  |  AMP dtype: {AMP_DTYPE}  |  AMP: {USE_AMP}')

# ── 공통 레이어 ───────────────────────────────────────────────
class _FFN(nn.Module):
    def __init__(self, d, do):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, d*4), nn.GELU(), nn.Dropout(do), nn.Linear(d*4, d))
    def forward(self, x): return self.net(x)

class _ITLayer(nn.Module):
    def __init__(self, d, h, do):
        super().__init__()
        self.n1 = nn.LayerNorm(d); self.n2 = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, h, dropout=do, batch_first=True)
        self.ff = _FFN(d, do); self.drop = nn.Dropout(do)
    def forward(self, x):
        a, _ = self.attn(x, x, x)
        x = self.n1(x + self.drop(a))
        return self.n2(x + self.drop(self.ff(x)))

# ── ① iTransformer ────────────────────────────────────────────
class iTransformer(nn.Module):
    def __init__(self, seq_len, n_features, d_model, n_heads, n_layers, dropout):
        super().__init__()
        self.feat_proj   = nn.Linear(seq_len, d_model)
        self.feat_layers = nn.ModuleList([_ITLayer(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.feat_norm   = nn.LayerNorm(d_model)
        self.coin_layers = nn.ModuleList([_ITLayer(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.coin_norm   = nn.LayerNorm(d_model)

    def forward(self, x):
        B, T, N, F = x.shape
        x_feat      = x.permute(0, 2, 3, 1).reshape(B * N, F, T)
        feat_tokens = self.feat_proj(x_feat)
        for l in self.feat_layers: feat_tokens = l(feat_tokens)
        feat_tokens = self.feat_norm(feat_tokens)
        coin_repr   = feat_tokens.mean(dim=1).reshape(B, N, -1)
        for l in self.coin_layers: coin_repr = l(coin_repr)
        coin_repr = self.coin_norm(coin_repr)
        v_emb     = coin_repr.reshape(B, -1)
        return v_emb, coin_repr

# ── ② NewsEncoder ─────────────────────────────────────────────
class NewsEncoder(nn.Module):
    """(B, N, 4) → (B, N, D_MODEL)"""
    def __init__(self, n_news_feat, d_model, dropout):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(n_news_feat, d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model, d_model), nn.LayerNorm(d_model))
    def forward(self, nf): return self.proj(nf)

# ── ③ BiCrossAttn ─────────────────────────────────────────────
class BiCrossAttn(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn_v2l = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_l2v = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.combine  = nn.Linear(d_model * 2, d_model)
        self.norm1    = nn.LayerNorm(d_model)
        self.norm2    = nn.LayerNorm(d_model)
        self.ff       = _FFN(d_model, dropout)
        self.drop     = nn.Dropout(dropout)

    def forward(self, coin_repr, l_emb):
        out_v2l, _ = self.attn_v2l(coin_repr, l_emb, l_emb)
        out_l2v, _ = self.attn_l2v(l_emb, coin_repr, coin_repr)
        fused = self.combine(torch.cat([out_v2l, out_l2v], dim=-1))
        fused = self.norm1(coin_repr + self.drop(fused))
        return self.norm2(fused + self.drop(self.ff(fused)))

# ── ④ LLMReasoningModule ─────────────────────────────────────
class LLMReasoningModule(nn.Module):
    def __init__(self, v_dim=D_MODEL, llm_name=LLM_NAME, device='cpu', use_lora=True):
        super().__init__()
        self.device = device
        from transformers import AutoConfig, AutoTokenizer, AutoModel
        cfg = AutoConfig.from_pretrained(llm_name)
        self.hidden_size = cfg.hidden_size

        self.tokenizer = AutoTokenizer.from_pretrained(llm_name, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # A100: BF16 / 그 외 GPU: FP16
        llm_dtype = AMP_DTYPE if device != 'cpu' else torch.float32
        llm_base = AutoModel.from_pretrained(
            llm_name, trust_remote_code=True, torch_dtype=llm_dtype)

        if use_lora:
            from peft import get_peft_model, LoraConfig
            lora_cfg = LoraConfig(
                r=LORA_RANK, lora_alpha=LORA_ALPHA,
                target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
                lora_dropout=0.05, bias='none')
            self.llm = get_peft_model(llm_base, lora_cfg)
        else:
            self.llm = llm_base
        self.llm.to(device)

        self.v_proj = nn.Sequential(
            nn.Linear(v_dim, self.hidden_size), nn.GELU(),
            nn.Linear(self.hidden_size, self.hidden_size),
            nn.LayerNorm(self.hidden_size)).to(device)

    def forward(self, v_emb, news_texts):
        B = v_emb.size(0)
        v_tok = self.v_proj(v_emb.to(self.device)).unsqueeze(1)

        prompts = [f"{SYSTEM_PROMPT}\nNews: {t}\nAllocation:" for t in news_texts]
        enc = self.tokenizer(prompts, return_tensors='pt', padding=True,
                             truncation=True, max_length=256).to(self.device)
        text_embs = self.llm.get_input_embeddings()(enc.input_ids)

        combined = torch.cat([v_tok, text_embs], dim=1)
        mask = torch.cat([
            torch.ones(B, 1, device=self.device, dtype=torch.long),
            enc.attention_mask], dim=1)

        out = self.llm(inputs_embeds=combined, attention_mask=mask)
        return out.last_hidden_state[:, 0, :].float()

# ── ⑤ ActionHead (순수 softmax) ──────────────────────────────
class ActionHead(nn.Module):
    def __init__(self, in_dim, n_assets, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(512, 256),   nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, n_assets + 1))

    def forward(self, h):
        return F.softmax(self.net(h), dim=-1)

# ── ⑥ PriceHead ───────────────────────────────────────────────
class PriceHead(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, 64), nn.GELU(), nn.Linear(64, 2))
    def forward(self, cr): return self.net(cr)

# ── FinBERT inference 전용 ─────────────────────────────────────
class FinBERTEncoder:
    """inference 시 raw 헤드라인 → (N, 4) [pos,neg,neu,score]"""
    def __init__(self, device='cpu', cache_dir=None):
        from transformers import pipeline
        self.pipe = pipeline(
            'sentiment-analysis', model='ProsusAI/finbert',
            tokenizer='ProsusAI/finbert',
            device=0 if device == 'cuda' else -1,
            model_kwargs={'cache_dir': cache_dir} if cache_dir else {})
    def encode(self, headlines_per_coin: dict) -> torch.Tensor:
        out = []
        for coin in ASSETS:
            texts = headlines_per_coin.get(coin, [])
            if not texts:
                out.append([0.0, 0.0, 1.0, 0.0]); continue
            results = self.pipe(texts[:20], truncation=True, max_length=128)
            pos = float(np.mean([r['score'] if r['label']=='positive' else 0. for r in results]))
            neg = float(np.mean([r['score'] if r['label']=='negative' else 0. for r in results]))
            neu = float(np.mean([r['score'] if r['label']=='neutral'  else 0. for r in results]))
            out.append([pos, neg, neu, pos - neg])
        return torch.tensor(out, dtype=torch.float32)

# ── VLA 전체 모델 ─────────────────────────────────────────────
class VLAModel(nn.Module):
    def __init__(self, n_features=110, use_llm=False, hf_cache=None):
        super().__init__()
        V_DIM = N_ASSETS * D_MODEL
        self.enc        = iTransformer(SEQ_LEN, n_features, D_MODEL, N_HEADS, N_LAYERS, DROPOUT)
        self.enc_norm   = nn.LayerNorm(D_MODEL)
        self.news_enc   = NewsEncoder(NEWS_N_FEAT, D_MODEL, DROPOUT)
        self.news_norm  = nn.LayerNorm(D_MODEL)
        self.bi_cross   = BiCrossAttn(D_MODEL, N_HEADS, DROPOUT)
        self.p1_proj    = nn.Sequential(
            nn.Linear(V_DIM, V_DIM * 2), nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(V_DIM * 2, V_DIM))
        self.p1_head    = ActionHead(V_DIM, N_ASSETS, DROPOUT)
        self.price_head = PriceHead(D_MODEL)
        self.llm_module  = None
        self.action_head = None
        if use_llm:
            self._load_llm(hf_cache)

    def _load_llm(self, hf_cache=None):
        if hf_cache:
            import os; os.environ['TRANSFORMERS_CACHE'] = hf_cache
        self.llm_module  = LLMReasoningModule(
            v_dim=D_MODEL, llm_name=LLM_NAME, device=device, use_lora=True)
        self.action_head = ActionHead(
            self.llm_module.hidden_size, N_ASSETS, DROPOUT).to(device)
        self.llm_module.llm.print_trainable_parameters()
        print(f'  LLM {LLM_NAME} + LoRA(r={LORA_RANK}, α={LORA_ALPHA}) 로드 완료')

    def forward(self, x, w, news_feat):
        return self.forward_phase1(x, w, news_feat)

    def forward_phase1(self, x, w, news_feat):
        B = x.shape[0]
        v_emb, coin_repr = self.enc(x)
        coin_repr = self.enc_norm(coin_repr)
        l_emb  = self.news_norm(self.news_enc(news_feat))
        fused  = self.bi_cross(coin_repr, l_emb)
        fv     = self.p1_proj(fused.reshape(B, -1))
        w_star = self.p1_head(fv)
        pp     = self.price_head(coin_repr)
        return w_star, fv, pp

    def forward_phase2(self, x, w, news_feat, news_texts):
        assert self.llm_module is not None, "_load_llm() 먼저 호출"
        B = x.shape[0]
        with torch.no_grad():
            v_emb, coin_repr = self.enc(x)
        coin_repr = self.enc_norm(coin_repr)
        l_emb  = self.news_norm(self.news_enc(news_feat))
        fused  = self.bi_cross(coin_repr, l_emb)
        v_pool = fused.mean(dim=1)
        hidden = self.llm_module(v_pool, news_texts)
        w_star = self.action_head(hidden)
        return w_star, fused.reshape(B, -1), None

    def freeze_encoder(self):
        for p in self.enc.parameters(): p.requires_grad_(False)
        print('  iTransformer 동결 완료')

    def trainable_params_phase1(self):
        return (list(self.enc.parameters()) + list(self.enc_norm.parameters()) +
                list(self.news_enc.parameters()) + list(self.news_norm.parameters()) +
                list(self.bi_cross.parameters()) + list(self.p1_proj.parameters()) +
                list(self.p1_head.parameters()) + list(self.price_head.parameters()))

    def trainable_params_phase2(self):
        ps = (list(self.news_enc.parameters()) + list(self.news_norm.parameters()) +
              list(self.bi_cross.parameters()))
        if self.llm_module:
            ps += [p for p in self.llm_module.parameters() if p.requires_grad]
            ps += list(self.action_head.parameters())
        return ps

def make_model(n_features, use_llm=False, hf_cache=None):
    return VLAModel(n_features, use_llm=use_llm, hf_cache=hf_cache).to(device)

# 파라미터 확인
_m = make_model(110, use_llm=False)
_total = sum(p.numel() for p in _m.parameters())
_enc   = sum(p.numel() for p in _m.enc.parameters())
del _m
print('모델 정의 완료  [v16]')
print(f'  NEWS_N_FEAT={NEWS_N_FEAT}  BATCH_SIZE={BATCH_SIZE}  GRPO_BATCH={GRPO_BATCH_SIZE}')
print(f'  AMP={USE_AMP}  dtype={AMP_DTYPE}')
print(f'  파라미터(LLM제외): 전체={_total:,}  인코더={_enc:,}')
print(f'  EPOCHS_G={EPOCHS_G}  WF_ROUNDS={WF_ROUNDS}  ROUND_WEEKS={ROUND_WEEKS}')


Device: cuda  |  AMP dtype: torch.bfloat16  |  AMP: True
모델 정의 완료  [v16]
  NEWS_N_FEAT=4  BATCH_SIZE=64  GRPO_BATCH=16
  AMP=True  dtype=torch.bfloat16
  파라미터(LLM제외): 전체=11,791,304  인코더=3,175,680
  EPOCHS_G=20  WF_ROUNDS=10  ROUND_WEEKS=1


In [5]:
# ══════════════════════════════════════════════════════════════
#  셀 2/5  —  Hindsight Sortino GT  [v16]
#  멀티 호라이즌: 12스텝(단기) + 36스텝(중기) 블렌드
#  Turnover 스무딩 + 낙폭(drawdown) 기반 현금 신호 (완화)
# ══════════════════════════════════════════════════════════════
import numpy as np

SHARPE_HORIZON = 12
LONG_HORIZON   = 36
GT_BLEND       = 0.6
SMOOTH_ALPHA   = 0.35

def _sortino_raw(prices, horizon, n_assets=5):
    T = len(prices) - 1
    labels = np.zeros((T, n_assets + 1), dtype=np.float32)
    for t in range(T):
        end  = min(t + horizon, T)
        p    = prices[t:end + 1]
        rets = (p[1:] - p[:-1]) / (p[:-1] + 1e-9)
        mu           = rets.mean(axis=0)
        downside     = np.minimum(rets, 0.0)
        downside_dev = np.sqrt(np.mean(downside**2, axis=0)) + 1e-8
        sortino      = mu / downside_dev
        market_trend = rets.mean()
        if market_trend > 0:
            cash_score = 0.0
        else:
            base_cash = (abs(market_trend) / (rets.std() + 1e-8)) * 2.0
            dd = np.maximum(0, (p[0] - p.min(axis=0)) / (p[0] + 1e-9)).mean()
            dd_boost  = float(np.clip(dd / 0.05, 1.0, 2.0))
            cash_score = base_cash * dd_boost
        sv  = np.append(sortino, cash_score)
        pos = np.maximum(sv, 0.0)
        if pos.sum() < 1e-8:
            labels[t, -1] = 1.0
        else:
            labels[t] = pos / pos.sum()
    return labels

def generate_sortino_labels(prices, horizon=SHARPE_HORIZON, n_assets=5,
                             long_horizon=LONG_HORIZON, blend=GT_BLEND,
                             smooth_alpha=SMOOTH_ALPHA):
    raw_short = _sortino_raw(prices, horizon,      n_assets)
    raw_long  = _sortino_raw(prices, long_horizon, n_assets)
    blended = blend * raw_short + (1.0 - blend) * raw_long
    s = blended.sum(axis=1, keepdims=True)
    blended = np.where(s < 1e-8,
                       np.eye(n_assets + 1)[[-1]].repeat(len(blended), axis=0),
                       blended / s)
    labels = blended.copy()
    for t in range(1, len(labels)):
        labels[t] = (1.0 - smooth_alpha) * labels[t-1] + smooth_alpha * blended[t]
        s = labels[t].sum()
        labels[t] = labels[t] / s if s > 1e-8 else np.eye(n_assets + 1)[-1]
    return labels.astype(np.float32)

def verify_sortino_labels(labels, prices, commission=0.015, verbose=True):
    n = labels.shape[1] - 1
    e = 1.0; prev = np.zeros(n + 1); prev[-1] = 1.0
    for t in range(len(labels)):
        w    = labels[t]
        rets = (prices[t + 1] - prices[t]) / (prices[t] + 1e-9)
        cost = commission * float(np.abs(w - prev).sum())
        e *= (1 + float(np.dot(w[:n], rets)) - cost); prev = w.copy()
    btc      = float((prices[-1, 0] - prices[0, 0]) / (prices[0, 0] + 1e-9))
    traj_ret = e - 1.0
    if verbose:
        entropy  = -(labels * np.log(labels + 1e-8)).sum(axis=1).mean()
        turnover = np.abs(np.diff(labels, axis=0)).sum(axis=1).mean()
        print(f'  Sortino GT: {traj_ret*100:+.2f}%  BTC: {btc*100:+.2f}%  알파: {(traj_ret-btc)*100:+.2f}%')
        print(f'  entropy={entropy:.3f}  turnover_avg={turnover:.3f}')
    return traj_ret, btc

print('Hindsight Sortino GT [v16] 정의 완료')
print(f'  단기 {SHARPE_HORIZON}스텝 + 중기 {LONG_HORIZON}스텝 블렌드 {GT_BLEND:.0%}/{1-GT_BLEND:.0%}')
print(f'  현금 신호 완화: base*2.0 / dd_boost=clip(dd/0.05, 1.0, 2.0)')


Hindsight Sortino GT [v16] 정의 완료
  단기 12스텝 + 중기 36스텝 블렌드 60%/40%
  현금 신호 완화: base*2.0 / dd_boost=clip(dd/0.05, 1.0, 2.0)


In [6]:
# ══════════════════════════════════════════════════════════════
#  셀 3/5  —  Phase1Trainer  [v16]
#  입력: x (B,T,N,F) + news_feat (B,N,4)  ← pos/neg/neu/score
#  loss = KLD + rank(0.05) + VICReg(0.05) + PriceLoss(0.3)
#  AMP (BF16/FP16) 지원
# ══════════════════════════════════════════════════════════════
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
import numpy as np, torch, torch.nn as nn

class Phase1Trainer:
    def __init__(self, model, lr=3e-4, commission=0.001, device='cpu',
                 asset_names=None, rank_weight=0.05, vic_weight=0.05):
        self.model = model; self.device = device; self.commission = commission
        self.rw = rank_weight; self.vw = vic_weight
        names = asset_names or [f'coin{i}' for i in range(N_ASSETS)]
        self.names = names if names[-1] == '현금' else names + ['현금']
        self.n_out = N_ASSETS + 1
        self.opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        # AMP: BF16은 GradScaler 불필요, FP16은 필요
        self.use_amp  = USE_AMP
        self.amp_dtype = AMP_DTYPE
        self.scaler   = (torch.amp.GradScaler('cuda')
                         if USE_AMP and AMP_DTYPE == torch.float16
                         else None)

    @staticmethod
    def _kld(pred, target, fw):
        loss = F.kl_div(torch.log(pred.clamp(1e-8)), target, reduction='none').sum(dim=-1)
        return (loss * fw).mean()

    @staticmethod
    def _rk(fused_v, target):
        regime = (target[:, -1] > 0.5).float()
        vn  = F.normalize(fused_v, dim=-1)
        sim = vn @ vn.T
        same = (regime[:, None] == regime[None, :]).float()
        return F.relu(1.0 - sim * (same * 2 - 1)).mean()

    @staticmethod
    def _vic(z, lv=1.0, lc=0.04, eps=1e-4):
        B, D = z.shape
        std = torch.sqrt(z.var(dim=0) + eps)
        vl  = F.relu(1.0 - std).mean()
        zc  = z - z.mean(dim=0)
        cov = (zc.T @ zc) / (B - 1)
        return lv * vl + lc * cov.pow(2).triu(1).sum() / D

    def _price_loss(self, price_pred, prices, idx_b, horizon=12):
        prices_t = torch.FloatTensor(prices).to(self.device)
        losses = []
        for i, idx in enumerate(idx_b):
            idx = int(idx.item())
            end = min(idx + horizon, len(prices) - 1)
            if end <= idx: continue
            fut  = prices_t[idx:end + 1]; curr = prices_t[idx]
            fret = (fut - curr) / (curr + 1e-9)
            tgt  = torch.stack([fret.min(dim=0).values, fret.max(dim=0).values], dim=-1)
            losses.append(F.mse_loss(price_pred[i].float(), tgt))
        return torch.stack(losses).mean() if losses else torch.tensor(0.0, device=self.device)

    def _equity(self, X, news_X, prices):
        self.model.eval()
        e = 1.0; prev = np.zeros(self.n_out); prev[-1] = 1.0
        wc = torch.zeros(1, self.n_out, device=self.device); wc[0, -1] = 1.0
        with torch.no_grad():
            for i in range(len(X) - 1):
                xb = torch.FloatTensor(X[i]).unsqueeze(0).to(self.device)
                nb = torch.FloatTensor(news_X[i]).unsqueeze(0).to(self.device)
                with torch.amp.autocast('cuda', enabled=self.use_amp, dtype=self.amp_dtype):
                    w, _, _ = self.model(xb, wc, nb)
                w = w.squeeze(0).float().cpu().numpy()
                w = np.clip(w, 0, 1); w /= (w.sum() + 1e-8)
                r = (prices[i + 1] - prices[i]) / (prices[i] + 1e-9)
                e *= (1 + np.dot(w[:N_ASSETS], r) - self.commission * np.abs(w - prev).sum())
                prev = w
                wc = torch.FloatTensor(w).unsqueeze(0).to(self.device)
        self.model.train()
        return e - 1.0

    def train(self, X, news_X, oracle_labels, focal_weights, prices,
              n_epochs=80, batch_size=32, log_interval=20,
              save_path=None, round_name=''):
        indices = torch.arange(len(X))
        ds = TensorDataset(
            torch.FloatTensor(X),              # (B,T,N,F)
            torch.FloatTensor(news_X),         # (B,N,4)
            torch.FloatTensor(oracle_labels),  # (B,N+1)
            torch.FloatTensor(focal_weights),  # (B,)
            indices)
        dl  = DataLoader(ds, batch_size=batch_size, shuffle=False,
                         drop_last=True, pin_memory=(self.device=='cuda'),
                         num_workers=2)
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=n_epochs)
        btc = (prices[-1, 0] / prices[0, 0] - 1)
        print(f'  loss=KLD+{self.rw}*rank+{self.vw}*vic+0.3*price  BTC:{btc*100:+.2f}%  AMP:{self.use_amp}({self.amp_dtype})')
        prev_wc = torch.zeros(1, self.n_out, device=self.device); prev_wc[0, -1] = 1.0
        best = float('inf')

        for ep in range(n_epochs):
            tk = tr = tv = tp = 0.0; nb_cnt = 0
            for xb, nb, yb, fw, idx_b in dl:
                xb = xb.to(self.device, non_blocking=True)
                nb = nb.to(self.device, non_blocking=True)
                yb = yb.to(self.device, non_blocking=True)
                fw = fw.to(self.device, non_blocking=True)
                B  = xb.shape[0]
                wc_b = prev_wc.expand(B, -1).detach()

                with torch.amp.autocast('cuda', enabled=self.use_amp, dtype=self.amp_dtype):
                    pred, fused_v, price_pred = self.model(xb, wc_b, nb)
                    kld  = self._kld(pred.float(), yb, fw)
                    rk   = self._rk(fused_v.float(), yb)
                    vic  = self._vic(fused_v.float())
                    pl   = self._price_loss(price_pred, prices, idx_b)
                    loss = kld + self.rw * rk + self.vw * vic + 0.3 * pl

                if not torch.isfinite(loss): continue
                self.opt.zero_grad()
                if self.scaler:
                    self.scaler.scale(loss).backward()
                    self.scaler.unscale_(self.opt)
                    nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    self.scaler.step(self.opt)
                    self.scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    self.opt.step()

                with torch.no_grad():
                    with torch.amp.autocast('cuda', enabled=self.use_amp, dtype=self.amp_dtype):
                        w_last, _, _ = self.model(xb[-1:], prev_wc, nb[-1:])
                prev_wc = w_last.float().detach()
                tk += kld.item(); tr += rk.item(); tv += vic.item(); tp += pl.item(); nb_cnt += 1

            sch.step()
            avg_kld = tk / max(nb_cnt, 1)
            if avg_kld < best:
                best = avg_kld
                if save_path:
                    torch.save(self.model.state_dict(), save_path)
            if (ep + 1) % log_interval == 0:
                mod_ret = self._equity(X, news_X, prices)
                wc_fix  = torch.zeros(1, self.n_out, device=self.device); wc_fix[0, -1] = 1.0
                n0 = torch.FloatTensor(news_X[-1]).unsqueeze(0).to(self.device)
                with torch.no_grad():
                    with torch.amp.autocast('cuda', enabled=self.use_amp, dtype=self.amp_dtype):
                        sp, _, _ = self.model(
                            torch.FloatTensor(X[-1]).unsqueeze(0).to(self.device), wc_fix, n0)
                    sp = sp.squeeze(0).float().cpu().numpy()
                ps = ' '.join(f'{n}={sp[i]*100:.0f}%' for i, n in enumerate(self.names))
                print(f'    ep {ep+1:3d}  kld={avg_kld:.5f}  rk={tr/max(nb_cnt,1):.5f}  price={tp/max(nb_cnt,1):.5f}')
                print(f'            모델수익:{mod_ret*100:+.2f}%  BTC:{btc*100:+.2f}%')
                print(f'            pred=[{ps}]')
        return self._equity(X, news_X, prices), btc

print('Phase1Trainer v16 정의 완료')
print(f'  AMP={USE_AMP}  dtype={AMP_DTYPE}  pin_memory=True  num_workers=2')
print('  loss = KLD + rank*0.05 + vic*0.05 + price*0.3')


Phase1Trainer v16 정의 완료
  AMP=True  dtype=torch.bfloat16  pin_memory=True  num_workers=2
  loss = KLD + rank*0.05 + vic*0.05 + price*0.3


In [7]:
# ── 셀 4 : 데이터 준비 + Phase1 Walk-Forward 추론  (v17) ──────────────────────
#   Phase 1은 재학습 없음 - 기존 v16 체크포인트 로드 후 추론만 수행
_VER = 'v17'
import numpy as np, pandas as pd, torch, os

# ── 심볼 매핑 (parquet: BTCUSDT → 뉴스CSV: BTC) ──────────────────────────────
USDT_TO_SHORT = {
    'BTCUSDT':'BTC','ETHUSDT':'ETH','SOLUSDT':'SOL',
    'XRPUSDT':'XRP','DOGEUSDT':'DOGE'
}

# ── 1. OHLCV 데이터 로드 ─────────────────────────────────────────────────────
print("Loading OHLCV parquet...")
df = pd.read_parquet(DATASET_FILE)
df['open_time'] = pd.to_datetime(df['open_time'])
df = df.sort_values(['open_time','symbol']).reset_index(drop=True)
symbols       = sorted(df['symbol'].unique().tolist())
N_COINS       = len(symbols)
short_symbols = [USDT_TO_SHORT.get(s, s) for s in symbols]
print(f"Symbols(USDT): {symbols}")
print(f"Symbols(short): {short_symbols}")

NON_FEAT  = {'symbol','open_time','date','target','symbol_encoded'}
feat_cols = [c for c in df.columns if c not in NON_FEAT]
F_DIM     = len(feat_cols)
print(f"Feature cols ({F_DIM}): {feat_cols[:8]}...")

pivot = df.pivot(index='open_time', columns='symbol', values=feat_cols)
pivot.columns = [f"{col[1]}_{col[0]}" for col in pivot.columns]
pivot = pivot.sort_index().ffill().bfill()
ts_index = pivot.index
T_total  = len(ts_index)

arr = np.zeros((T_total, N_COINS, F_DIM), dtype=np.float32)
for i, sym in enumerate(symbols):
    for j, fc in enumerate(feat_cols):
        col = f"{sym}_{fc}"
        if col in pivot.columns:
            arr[:, i, j] = pivot[col].values
print(f"OHLCV array: {arr.shape}")

# ── 2. news_sentiment_8h.csv 로드 ────────────────────────────────────────────
print("Loading news_sentiment_8h.csv...")
ns_df   = pd.read_csv(f'{DATA_DIR}/news_sentiment_8h.csv')
sym_col = 'coin' if 'coin' in ns_df.columns else 'symbol'

ns_df['start_hour'] = ns_df['bucket_8h'].str.extract(r'^(\d+)h')[0].astype(int)
ns_df['ts_bucket']  = (pd.to_datetime(ns_df['date']) +
                        pd.to_timedelta(ns_df['start_hour'], unit='h'))

ns_pivot_list = []
for feat in NEWS_FEATURE_NAMES:
    pv = ns_df.pivot_table(index='ts_bucket', columns=sym_col,
                            values=feat, aggfunc='mean')
    pv = pv.reindex(columns=short_symbols).sort_index()
    ns_pivot_list.append(pv)

ns_ts     = ns_pivot_list[0].index
common_ts = ts_index.intersection(ns_ts)
print(f"Common timestamps: {len(common_ts)}")

arr      = arr[ts_index.isin(common_ts)]
ts_index = ts_index[ts_index.isin(common_ts)]

l_arr = np.stack([pv.loc[common_ts].fillna(0).values for pv in ns_pivot_list], axis=-1)
l_arr = l_arr.astype(np.float32)
print(f"Sentiment array: {l_arr.shape}  (expected last dim={NEWS_N_FEAT})")
assert l_arr.shape[-1] == NEWS_N_FEAT

# ── 3. news_all_8h.csv 텍스트 맵 (Phase 2 용) ────────────────────────────────
print("Loading news_all_8h.csv for text map...")
na_df      = pd.read_csv(f'{DATA_DIR}/news_all_8h.csv')
na_sym_col = 'coin' if 'coin' in na_df.columns else 'symbol'
na_time_col = 'published_at' if 'published_at' in na_df.columns else 'timestamp'
na_df[na_time_col] = pd.to_datetime(na_df[na_time_col])
na_df['ts_bucket'] = na_df[na_time_col].dt.floor('8h')

text_map_raw = {}
for _, row in na_df.iterrows():
    ts  = row['ts_bucket']
    sym = str(row.get(na_sym_col, 'ALL'))
    txt = str(row.get('text', row.get('title', row.get('headline', ''))))
    text_map_raw.setdefault(ts, {}).setdefault(sym, []).append(txt)
text_map_flat = {
    ts: {sym: ' '.join(txts)[:512] for sym, txts in sym_dict.items()}
    for ts, sym_dict in text_map_raw.items()
}
print(f"text_map entries: {len(text_map_flat)}")

# ── 4. Sliding window ─────────────────────────────────────────────────────────
X_wins = np.lib.stride_tricks.sliding_window_view(arr, window_shape=SEQ_LEN, axis=0)
X_wins = X_wins.transpose(0, 3, 1, 2)   # (W, SEQ_LEN, N, F)
N_wins = l_arr[SEQ_LEN-1:]              # (W, N, 4)
win_ts = ts_index[SEQ_LEN-1:]
W      = len(win_ts)
assert len(X_wins) == W == len(N_wins), f"mismatch X={len(X_wins)} N={len(N_wins)}"
print(f"Windows:{W}  X_wins:{X_wins.shape}  N_wins:{N_wins.shape}")

close_idx = feat_cols.index('close') if 'close' in feat_cols else 0
P_all  = arr[:, :, close_idx]
P_wins = P_all[SEQ_LEN-1:]

# ── 5. Walk-Forward 분할 함수 ─────────────────────────────────────────────────
all_dates = pd.DatetimeIndex(win_ts)

def get_round_split(r, all_dates, round_weeks=ROUND_WEEKS):
    te_end   = all_dates[-1] - pd.DateOffset(weeks=round_weeks * (WF_ROUNDS - 1 - r))
    te_start = te_end - pd.DateOffset(weeks=round_weeks) + pd.Timedelta(hours=8)
    tr_end   = te_start - pd.Timedelta(hours=8)
    return all_dates <= tr_end, (all_dates >= te_start) & (all_dates <= te_end)

# Phase 1 체크포인트 스마트 매핑 (기존 v16 5개 ckpt → 10라운드에 재사용)
P1_CKPT_DATES = {
    1: pd.Timestamp('2025-03-01'),
    2: pd.Timestamp('2025-06-01'),
    3: pd.Timestamp('2025-09-01'),
    4: pd.Timestamp('2025-12-01'),
    5: pd.Timestamp('2026-03-01'),
}
def get_best_p1_ckpt(test_start_ts, ver='v16'):
    best = 1
    for rnd, ts in P1_CKPT_DATES.items():
        if ts <= pd.Timestamp(test_start_ts):
            best = rnd
    return f'{CKPT_DIR}/p1_r{best}_{ver}.pt'

# ── 6. Phase 1 추론 (10라운드, 재학습 없음) ───────────────────────────────────
results = []
for rnd in range(WF_ROUNDS):
    print(f"\n{'='*60}")
    print(f"  Walk-Forward Round {rnd+1}/{WF_ROUNDS}")
    tr_mask, te_mask = get_round_split(rnd, all_dates)
    if tr_mask.sum() < 50 or te_mask.sum() < 5:
        print(f"  Skipping (train={tr_mask.sum()}, test={te_mask.sum()})")
        continue
    print(f"  Train:{tr_mask.sum()}  Test:{te_mask.sum()}")

    X_te  = X_wins[te_mask]
    N_te  = N_wins[te_mask]
    P_te  = P_wins[te_mask]
    ts_te = all_dates[te_mask]

    model_p1 = VLAModel(n_features=F_DIM, use_llm=False).to(DEVICE)
    ckpt_p1  = get_best_p1_ckpt(ts_te[0])
    model_p1.load_state_dict(torch.load(ckpt_p1, map_location=DEVICE))
    print(f"  Phase1 체크포인트 로드: {ckpt_p1}")

    model_p1.eval()
    w_preds = []
    with torch.no_grad():
        for i in range(0, len(X_te), BATCH_SIZE):
            xb = torch.tensor(X_te[i:i+BATCH_SIZE], dtype=torch.float32).to(DEVICE)
            nb = torch.tensor(N_te[i:i+BATCH_SIZE], dtype=torch.float32).to(DEVICE)
            wc = torch.zeros(xb.size(0), N_COINS + 1, device=DEVICE)
            wc[:, -1] = 1.0
            w_star, _, _ = model_p1.forward_phase1(xb, wc, nb)
            w_preds.append(w_star.cpu().numpy())
    w_preds = np.concatenate(w_preds, axis=0)

    rets      = np.diff(np.log(P_te + 1e-9), axis=0)
    port_rets = (w_preds[:-1, :N_COINS] * rets).sum(axis=1)
    cum_port  = np.exp(np.cumsum(port_rets)) - 1
    cum_eq    = np.exp(np.cumsum(rets.mean(axis=1))) - 1
    neg_std   = port_rets[port_rets < 0].std() if (port_rets < 0).any() else 1e-9
    sortino   = port_rets.mean() / (neg_std + 1e-9) * np.sqrt(365 * 3)

    print(f"  Sortino:{sortino:.3f}  CumRet:{cum_port[-1]*100:.1f}%  Bench:{cum_eq[-1]*100:.1f}%")
    results.append({'round': rnd+1, 'sortino': sortino,
                    'cum_ret': cum_port[-1], 'eq_bench': cum_eq[-1],
                    'ts_te': ts_te, 'ckpt_p1': ckpt_p1})

print(f"\n{'='*60}  Walk-Forward Phase1 Summary")
for r in results:
    print(f"  R{r['round']}: Sortino={r['sortino']:.3f}  CumRet={r['cum_ret']*100:.1f}%")
if results:
    print(f"  Avg Sortino: {np.mean([r['sortino'] for r in results]):.3f}")


Loading OHLCV parquet...
Symbols(USDT): ['BTCUSDT', 'DOGEUSDT', 'ETHUSDT', 'SOLUSDT', 'XRPUSDT']
Symbols(short): ['BTC', 'DOGE', 'ETH', 'SOL', 'XRP']
Feature cols (111): ['open', 'close', 'funding_rate', 'rsi_7', 'rsi_14', 'rsi_21', 'macd_hist_norm', 'bb_pct']...
OHLCV array: (2316, 5, 111)
Loading news_sentiment_8h.csv...
Common timestamps: 2316
Sentiment array: (2316, 5, 4)  (expected last dim=4)
Loading news_all_8h.csv for text map...
text_map entries: 2317
Windows:2257  X_wins:(2257, 60, 5, 111)  N_wins:(2257, 5, 4)
Generating oracle labels...
oracle_labels:(2257, 6)  focal_weights:(2257,)

  Walk-Forward Round 1/10
  Train:2047  Test:21
  Phase1 체크포인트 로드: /content/drive/MyDrive/X-MultiVLA_rev5/checkpoints/p1_r4_v16.pt
  Sortino:-10.264  CumRet:-7.5%  Bench:-13.7%

  Walk-Forward Round 2/10
  Train:2068  Test:21
  Phase1 체크포인트 로드: /content/drive/MyDrive/X-MultiVLA_rev5/checkpoints/p1_r4_v16.pt
  Sortino:3.790  CumRet:1.2%  Bench:2.5%

  Walk-Forward Round 3/10
  Train:2089  Test:21

In [8]:
# ── 셀 5 : GRPO Phase 2  (v17) ───────────────────────────────────────────────
_VER_G = 'v17_grpo'

import subprocess, sys
# torchao 버전 확인: 이미 >= 0.16.0이면 재로드 금지 (C++ op 이중 등록 방지)
try:
    import importlib.metadata
    from packaging.version import Version
    _tv = Version(importlib.metadata.version('torchao'))
    _needs_upgrade = _tv < Version('0.16.0')
except Exception:
    _needs_upgrade = True

if _needs_upgrade:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchao>=0.16.0'], check=True)
    for mod in list(sys.modules.keys()):
        if 'peft' in mod or 'torchao' in mod:
            del sys.modules[mod]
    print("torchao 업그레이드 완료, peft/torchao 재로드")
else:
    # 이미 0.16.0 이상 → torchao는 그대로, peft만 재로드
    for mod in list(sys.modules.keys()):
        if 'peft' in mod:
            del sys.modules[mod]
    print(f"torchao {_tv} (>=0.16.0 확인) - peft만 재로드")

import copy, torch, torch.nn.functional as F, numpy as np

class GRPOTrainer:
    def __init__(self, model, ref_model, lr=GRPO_LR, kl_coef=0.1, device=DEVICE):
        self.model     = model
        self.ref_model = ref_model
        self.kl_coef   = kl_coef
        self.device    = device
        self.use_amp   = USE_AMP
        self.amp_dtype = AMP_DTYPE
        self.scaler = (torch.amp.GradScaler('cuda')
                       if USE_AMP and AMP_DTYPE == torch.float16 else None)

        params = (list(model.bi_cross.parameters()) +
                  list(model.news_enc.parameters()) +
                  list(model.enc_norm.parameters()) +
                  list(model.news_norm.parameters()) +
                  list(model.action_head.parameters()))
        if model.llm_module is not None:
            params += [p for p in model.llm_module.parameters() if p.requires_grad]
        self.opt = torch.optim.AdamW(params, lr=lr)

    @torch.no_grad()
    def _ref_logw(self, xb, wc, nb, texts):
        with torch.amp.autocast('cuda', enabled=self.use_amp, dtype=self.amp_dtype):
            w_ref, _, _ = self.ref_model.forward_phase2(xb, wc, nb, texts)
        return torch.log(w_ref.float() + 1e-9)

    def _reward(self, w, prices_next, prices_cur):
        ret       = prices_next / (prices_cur + 1e-9) - 1.0
        port_ret  = (w[:, :N_COINS] * ret).sum(dim=1)
        bench_ret = ret.mean(dim=1)   # 동일비중 벤치마크
        active    = port_ret - bench_ret
        return active.clamp(-0.1, 0.1)

    def grpo_step(self, xb, wc, nb, texts, prices_cur, prices_next):
        xb          = xb.to(self.device)
        wc          = wc.to(self.device)
        nb          = nb.to(self.device)
        prices_cur  = prices_cur.to(self.device)
        prices_next = prices_next.to(self.device)

        with torch.amp.autocast('cuda', enabled=self.use_amp, dtype=self.amp_dtype):
            w_new, _, _ = self.model.forward_phase2(xb, wc, nb, texts)

        log_w_new = torch.log(w_new.float() + 1e-9)
        log_w_ref = self._ref_logw(xb, wc, nb, texts)

        reward = self._reward(w_new.float(), prices_next, prices_cur)
        kl     = (w_new.float() * (log_w_new - log_w_ref)).sum(dim=1).mean()
        loss   = -(reward.mean() - self.kl_coef * kl)

        self.opt.zero_grad()
        if self.scaler:
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(self.opt)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.scaler.step(self.opt)
            self.scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.opt.step()
        return loss.item(), reward.mean().item(), kl.item()

    def train(self, X, N_feat, P_wins, win_ts,
              epochs=EPOCHS_G, batch_size=GRPO_BATCH_SIZE):
        W = len(X)
        for ep in range(epochs):
            idx  = np.arange(W - 1)
            ep_l, ep_r, kl_v = [], [], 0.0
            for start in range(0, len(idx), batch_size):
                bi  = idx[start:start+batch_size]
                xb  = torch.tensor(X[bi],      dtype=torch.float32)
                nb  = torch.tensor(N_feat[bi], dtype=torch.float32)
                wc  = torch.zeros(len(bi), N_COINS + 1)
                wc[:, -1] = 1.0
                pc  = torch.tensor(P_wins[bi], dtype=torch.float32)
                future_idx = np.minimum(bi + 12, len(P_wins) - 1)
                pn  = torch.tensor(P_wins[future_idx], dtype=torch.float32)

                texts = []
                for i in bi:
                    ts        = win_ts[i]
                    sym_texts = text_map_flat.get(ts, {})
                    combined  = (' | '.join(f"{s}: {t}" for s, t in sym_texts.items())[:1024]
                                 if sym_texts else "No news available.")
                    texts.append(combined)

                lv, rv, kl_v = self.grpo_step(xb, wc, nb, texts, pc, pn)
                ep_l.append(lv); ep_r.append(rv)

            if (ep + 1) % max(1, epochs // 5) == 0:
                print(f"    GRPO ep {ep+1}/{epochs}  "
                      f"loss={np.mean(ep_l):.4f}  reward={np.mean(ep_r):.4f}  kl={kl_v:.4f}")


# ── Walk-Forward Phase 2 ──────────────────────────────────────────────────────
print("Starting GRPO Phase 2 Walk-Forward...")

for rnd_res in results:
    rnd = rnd_res['round']
    print(f"\n{'='*60}")
    print(f"  GRPO Round {rnd}/{WF_ROUNDS}")

    tr_mask, te_mask = get_round_split(rnd - 1, all_dates)
    X_tr  = X_wins[tr_mask];  X_te = X_wins[te_mask]
    N_tr  = N_wins[tr_mask];  N_te = N_wins[te_mask]
    P_tr  = P_wins[tr_mask];  P_te = P_wins[te_mask]
    ts_tr = all_dates[tr_mask]
    ts_te = all_dates[te_mask]

    model_p2 = VLAModel(n_features=F_DIM, use_llm=True, hf_cache=HF_CACHE).to(DEVICE)
    ckpt = torch.load(rnd_res['ckpt_p1'], map_location=DEVICE)
    model_p2.load_state_dict(ckpt, strict=False)
    model_p2.freeze_encoder()

    ref_model = copy.deepcopy(model_p2).to(DEVICE)
    for p in ref_model.parameters():
        p.requires_grad_(False)
    ref_model.eval()

    grpo = GRPOTrainer(model_p2, ref_model, lr=GRPO_LR, kl_coef=0.1, device=DEVICE)
    print(f"  AMP={grpo.use_amp}  dtype={grpo.amp_dtype}  epochs={EPOCHS_G}")
    grpo.train(X_tr, N_tr, P_tr, ts_tr, epochs=EPOCHS_G)

    ckpt_g = f'{CKPT_DIR}/grpo_r{rnd}_{_VER_G}.pt'
    torch.save(model_p2.state_dict(), ckpt_g)
    print(f"  GRPO ckpt saved: {ckpt_g}")

    model_p2.eval()
    w_preds2 = []
    with torch.no_grad():
        for i in range(0, len(X_te), GRPO_BATCH_SIZE):
            xb  = torch.tensor(X_te[i:i+GRPO_BATCH_SIZE], dtype=torch.float32).to(DEVICE)
            nb  = torch.tensor(N_te[i:i+GRPO_BATCH_SIZE], dtype=torch.float32).to(DEVICE)
            wc  = torch.zeros(xb.size(0), N_COINS + 1, device=DEVICE)
            wc[:, -1] = 1.0
            texts = []
            for j in range(i, min(i + GRPO_BATCH_SIZE, len(X_te))):
                ts        = ts_te[j]
                sym_texts = text_map_flat.get(ts, {})
                combined  = (' | '.join(f"{s}: {t}" for s, t in sym_texts.items())[:1024]
                             if sym_texts else "No news available.")
                texts.append(combined)
            with torch.amp.autocast('cuda', enabled=USE_AMP, dtype=AMP_DTYPE):
                w_star, _, _ = model_p2.forward_phase2(xb, wc, nb, texts)
            w_preds2.append(w_star.float().cpu().numpy())
    w_preds2 = np.concatenate(w_preds2, axis=0)

    rets2      = np.diff(np.log(P_te + 1e-9), axis=0)
    port_rets2 = (w_preds2[:-1, :N_COINS] * rets2).sum(axis=1)
    cum_port2  = np.exp(np.cumsum(port_rets2)) - 1
    neg_std2   = port_rets2[port_rets2 < 0].std() if (port_rets2 < 0).any() else 1e-9
    sortino2   = port_rets2.mean() / (neg_std2 + 1e-9) * np.sqrt(365 * 3)

    print(f"  Sortino(P2): {sortino2:.3f} | CumRet: {cum_port2[-1]*100:.1f}%")
    rnd_res['sortino_p2'] = sortino2
    rnd_res['cum_ret_p2'] = cum_port2[-1]
    rnd_res['ckpt_grpo']  = ckpt_g

print(f"\n{'='*60}")
print("Final Walk-Forward Summary (Phase1 vs Phase2)")
print(f"{'Round':>5} {'Sortino P1':>12} {'Sortino P2':>12} {'CumRet P2':>12}")
for r in results:
    s2 = r.get('sortino_p2', float('nan'))
    c2 = r.get('cum_ret_p2', float('nan'))
    print(f"  R{r['round']}   {r['sortino']:>10.3f}   {s2:>10.3f}   {c2*100:>10.1f}%")


torchao 업그레이드 완료, peft/torchao 재로드
Starting GRPO Phase 2 Walk-Forward...

  GRPO Round 1/10


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
  LLM Qwen/Qwen2.5-1.5B + LoRA(r=16, α=32) 로드 완료
  iTransformer 동결 완료
  AMP=True  dtype=torch.bfloat16  epochs=20
    GRPO ep 4/20  loss=0.0010  reward=-0.0005  kl=0.0026
    GRPO ep 8/20  loss=0.0008  reward=-0.0004  kl=0.0011
    GRPO ep 12/20  loss=0.0007  reward=-0.0006  kl=0.0008
    GRPO ep 16/20  loss=0.0007  reward=-0.0005  kl=0.0009
    GRPO ep 20/20  loss=0.0007  reward=-0.0005  kl=0.0012
  GRPO ckpt saved: /content/drive/MyDrive/X-MultiVLA_rev5/checkpoints/grpo_r1_v17_grpo.pt
  Sortino(P2): -9.151 | CumRet: -11.1%

  GRPO Round 2/10


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
  LLM Qwen/Qwen2.5-1.5B + LoRA(r=16, α=32) 로드 완료
  iTransformer 동결 완료
  AMP=True  dtype=torch.bfloat16  epochs=20
    GRPO ep 4/20  loss=0.0008  reward=-0.0003  kl=0.0034
    GRPO ep 8/20  loss=0.0007  reward=-0.0003  kl=0.0025
    GRPO ep 12/20  loss=0.0006  reward=-0.0003  kl=0.0022
    GRPO ep 16/20  loss=0.0006  reward=-0.0004  kl=0.0017
    GRPO ep 20/20  loss=0.0006  reward=-0.0004  kl=0.0009
  GRPO ckpt saved: /content/drive/MyDrive/X-MultiVLA_rev5/checkpoints/grpo_r8_v17_grpo.pt
  Sortino(P2): -4.919 | CumRet: -2.1%

  GRPO Round 9/10


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
  LLM Qwen/Qwen2.5-1.5B + LoRA(r=16, α=32) 로드 완료
  iTransformer 동결 완료
  AMP=True  dtype=torch.bfloat16  epochs=20
    GRPO ep 4/20  loss=0.0011  reward=-0.0005  kl=0.0019
    GRPO ep 8/20  loss=0.0008  reward=-0.0006  kl=0.0012
    GRPO ep 12/20  loss=0.0007  reward=-0.0005  kl=0.0015
    GRPO ep 16/20  loss=0.0007  reward=-0.0005  kl=0.0006
    GRPO ep 20/20  loss=0.0007  reward=-0.0005  kl=0.0007
  GRPO ckpt saved: /content/drive/MyDrive/X-MultiVLA_rev5/checkpoints/grpo_r9_v17_grpo.pt
  Sortino(P2): 0.304 | CumRet: 0.1%

  GRPO Round 10/10


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
  LLM Qwen/Qwen2.5-1.5B + LoRA(r=16, α=32) 로드 완료
  iTransformer 동결 완료
  AMP=True  dtype=torch.bfloat16  epochs=20
    GRPO ep 4/20  loss=0.0006  reward=0.0000  kl=0.0040
    GRPO ep 8/20  loss=0.0004  reward=-0.0001  kl=0.0006
    GRPO ep 12/20  loss=0.0003  reward=-0.0001  kl=0.0008
    GRPO ep 16/20  loss=0.0003  reward=-0.0002  kl=0.0005
    GRPO ep 20/20  loss=0.0002  reward=-0.0001  kl=0.0003
  GRPO ckpt saved: /content/drive/MyDrive/X-MultiVLA_rev5/checkpoints/grpo_r10_v17_grpo.pt
  Sortino(P2): 14.933 | CumRet: 4.1%

Final Walk-Forward Summary (Phase1 vs Phase2)
Round   Sortino P1   Sortino P2    CumRet P2
  R1      -10.264       -9.151        -11.1%
  R2        3.790        3.680          1.9%
  R3       -7.389       -7.509         -4.2%
  R4       -5.337       -5.103         -4.1%
  R5       -1.134       -1.206         -0.7%
  R6        8.647        7.822          4.0%
  R7        1.232        1.032

In [9]:
# ══════════════════════════════════════════════════════════════
#  셀 6  —  21-Timestamp Blind Inference Wrapper 검증 (v17 적용)
# ══════════════════════════════════════════════════════════════

# 🔥 [변경] 가장 성과가 좋았던 최종 Round 10 모델을 지정합니다.
EVAL_ROUND = 10

# 🚨 [치명적 버그 수정] v16 좀비 가중치가 아니라, 방금 구운 v17 야수 가중치를 로드합니다!
ckpt_path = f'{CKPT_DIR}/grpo_r{EVAL_ROUND}_v17_grpo.pt'
print(f"체크포인트 로드: {ckpt_path}")

eval_model = make_model(len(feat_cols), use_llm=True, hf_cache=HF_CACHE)
eval_model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
eval_model.eval()
print("모델 로드 완료 (eval 모드)")

class InferenceWrapper:
    def __init__(self, model, device, commission=COMMISSION,
                 top_k=3, confidence_threshold=0.10):
        self.model      = model
        self.device     = device
        self.commission = commission
        self.top_k      = top_k
        self.conf_th    = confidence_threshold

    @torch.no_grad()
    def step(self, x_t, n_t, prev_w, news_text):
        xb = torch.tensor(x_t, dtype=torch.float32).to(self.device)
        nb = torch.tensor(n_t, dtype=torch.float32).to(self.device)
        wc = torch.tensor(prev_w, dtype=torch.float32).unsqueeze(0).to(self.device)

        with torch.amp.autocast('cuda', enabled=USE_AMP, dtype=AMP_DTYPE):
            w_raw, _, price_pred = self.model.forward_phase2(xb, wc, nb, [news_text])

        # 모델이 내뱉은 Softmax 원본 값
        w_raw = w_raw.float().cpu().numpy()[0]

        # price_pred 예외 처리
        if price_pred is not None:
            price_pred = price_pred.float().cpu().numpy()[0]  # (N, 2)
        else:
            price_pred = np.zeros((N_COINS, 2))

        # ✅ 족쇄 해제된 날것의 확률 분포 분포를 최종 비중으로 사용
        final_w = w_raw

        return final_w, w_raw, price_pred


def run_validation_21(model, X_te, N_te, P_te, ts_te,
                       text_map_flat, n_eval=21):

    eval_start = len(X_te) - n_eval
    X_eval  = X_te[eval_start:]
    N_eval  = N_te[eval_start:]
    P_eval  = P_te[eval_start:]
    ts_eval = ts_te[eval_start:]

    print(f"\n{'='*70}")
    print(f"  21-Timestamp Blind Inference (v17 Dynamic Model)")
    print(f"  블라인드 기준시점: {ts_eval[0]}")
    print(f"  검증 마지막시점  : {ts_eval[-1]}")
    print(f"{'='*70}")

    wrapper = InferenceWrapper(eval_model, DEVICE)
    prev_w  = np.zeros(N_COINS + 1); prev_w[-1] = 1.0
    equity  = 1.0
    history = []

    for i in range(n_eval - 1):
        ts = ts_eval[i]

        x_t = X_eval[i:i+1]
        n_t = N_eval[i:i+1]

        sym_texts = text_map_flat.get(ts, {})
        news_text = (' | '.join(f"{s}: {t}" for s, t in sym_texts.items())[:1024]
                     if sym_texts else "No news available.")

        final_w, raw_w, pp = wrapper.step(x_t, n_t, prev_w, news_text)

        p_cur    = P_eval[i]
        p_next   = P_eval[i + 1]
        ret      = (p_next - p_cur) / (p_cur + 1e-9)
        cost     = COMMISSION * float(np.abs(final_w - prev_w).sum())
        step_ret = float(np.dot(final_w[:N_COINS], ret)) - cost
        equity  *= (1 + step_ret)

        action = short_symbols[np.argmax(final_w[:N_COINS])] \
                 if final_w[:N_COINS].max() > 0.1 else 'CASH'

        print(f"\n  Step {i+1:02d} | {ts} | ret={step_ret*100:+.2f}% | equity={equity:.4f}")
        print(f"  {'심볼':>6} {'현재가':>12} {'예상MIN':>12} {'예상MAX':>12} {'비중':>6}")
        for j, sym in enumerate(short_symbols):
            e_min = p_cur[j] * (1 + pp[j, 0])
            e_max = p_cur[j] * (1 + pp[j, 1])
            print(f"  {sym:>6} {p_cur[j]:>12.4f} {e_min:>12.4f} {e_max:>12.4f} {final_w[j]*100:>5.1f}%")
        print(f"  {'현금':>6} {'':>12} {'':>12} {'':>12} {final_w[-1]*100:>5.1f}%")

        history.append({
            'step': i+1, 'ts': ts, 'action': action,
            'weights': final_w, 'ret': step_ret, 'equity': equity
        })

        prev_w = final_w.copy()

    port_rets = np.array([h['ret'] for h in history])
    cum_ret   = equity - 1.0
    neg_std   = port_rets[port_rets < 0].std() \
                if (port_rets < 0).any() else 1e-9
    sortino   = port_rets.mean() / (neg_std + 1e-9) * np.sqrt(365 * 3)
    btc_ret   = float((P_eval[-1,0] - P_eval[0,0]) / (P_eval[0,0] + 1e-9))

    print(f"\n{'='*70}")
    print(f"  누적 수익률  : {cum_ret*100:+.2f}%")
    print(f"  BTC 벤치마크 : {btc_ret*100:+.2f}%")
    print(f"  알파         : {(cum_ret-btc_ret)*100:+.2f}%")
    print(f"  Sortino      : {sortino:.3f}")
    print(f"{'='*70}")

    return history, cum_ret, sortino


history_21, cum_21, sortino_21 = run_validation_21(
    eval_model, X_te, N_te, P_te, ts_te, text_map_flat
)

체크포인트 로드: /content/drive/MyDrive/X-MultiVLA_rev5/checkpoints/grpo_r10_v17_grpo.pt


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815
  LLM Qwen/Qwen2.5-1.5B + LoRA(r=16, α=32) 로드 완료
모델 로드 완료 (eval 모드)

  21-Timestamp Blind Inference (v17 Dynamic Model)
  블라인드 기준시점: 2026-04-05 00:00:00
  검증 마지막시점  : 2026-04-11 16:00:00

  Step 01 | 2026-04-05 00:00:00 | ret=-1.07% | equity=0.9893
      심볼          현재가        예상MIN        예상MAX     비중
     BTC   67177.0000   67177.0000   67177.0000  14.4%
    DOGE       0.0917       0.0917       0.0917  18.8%
     ETH    2061.2700    2061.2700    2061.2700  12.8%
     SOL      80.6500      80.6500      80.6500  13.8%
     XRP       1.3117       1.3117       1.3117  19.6%
      현금                                         20.6%

  Step 02 | 2026-04-05 08:00:00 | ret=+0.13% | equity=0.9906
      심볼          현재가        예상MIN        예상MAX     비중
     BTC   66892.5000   66892.5000   66892.5000  14.5%
    DOGE       0.0902       0.0902       0.0902  18.8%
     ETH    2038.0601    2038.0601    2038.0601  12.8%
     

In [ ]:
import os, subprocess, shutil
from getpass import getpass

# ── GitHub 설정 ───────────────────────────────────────────────
GITHUB_USER  = 'yimju'
GITHUB_REPO  = 'X-MultiVLA'
GITHUB_URL   = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git'
BRANCH       = 'main'

GIT_NAME  = input("Git 이름 (예: yimju): ").strip()
GIT_EMAIL = input("Git 이메일: ").strip()
PAT       = getpass("GitHub Personal Access Token (입력 숨김): ")

# ── 클론 ──────────────────────────────────────────────────────
os.chdir('/content')  # 작업 디렉토리 리셋 (필수 안전장치)
CLONE_DIR = '/content/X-MultiVLA'
if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)

auth_url = f'https://{PAT}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git'
subprocess.run(['git', 'clone', '-b', BRANCH, auth_url, CLONE_DIR], check=True)
print(f"✅ 클론 완료: {CLONE_DIR}")

# ── 🎯 현재 노트북 파일 '하나만' 복사 ──────────────────────────
DRIVE_DIR = '/content/drive/MyDrive/X-MultiVLA_rev5'

# 👇 여기에 지금 깃허브에 올릴 노트북 파일의 정확한 이름을 적어주세요!
TARGET_NOTEBOOK = 'X-MultiVLA_Rev5_v2_1week.ipynb'
src_file = os.path.join(DRIVE_DIR, TARGET_NOTEBOOK)

print(f"\n[노트북 파일 복사]")
if os.path.exists(src_file):
    dst = os.path.join(CLONE_DIR, TARGET_NOTEBOOK)
    shutil.copy2(src_file, dst)
    print(f"  복사 완료: {TARGET_NOTEBOOK}")
else:
    print(f"❌ 에러: 구글 드라이브에서 '{TARGET_NOTEBOOK}' 파일을 찾을 수 없습니다.")
    print("파일 이름을 다시 확인해 주세요.")
    raise SystemExit()

# ── 커밋 & 푸시 ───────────────────────────────────────────────
os.chdir(CLONE_DIR)
subprocess.run(['git', 'config', 'user.name',  GIT_NAME],  check=True)
subprocess.run(['git', 'config', 'user.email', GIT_EMAIL], check=True)
subprocess.run(['git', 'add', '.'], check=True)

status = subprocess.run(['git', 'status', '--short'], capture_output=True, text=True)
print("\n[Git Status]")
print(status.stdout if status.stdout.strip() else "변경사항 없음")

if status.stdout.strip():
    # 💡 커밋 메시지를 v2로 변경했습니다.
    commit_msg = 'Add X-MultiVLA Rev5 v2 notebook (Phase1+GRPO Phase2)'
    subprocess.run(['git', 'commit', '-m', commit_msg], check=True)
    subprocess.run(['git', 'push', auth_url, BRANCH], check=True)
    print(f"\n✅ 푸시 완료: {GITHUB_URL}")
else:
    print("변경사항 없음 — 이미 최신 상태입니다.")

Git 이름 (예: yimju): yimju
